# Lab 5, Day 1 — Exploration and Cleaning

Explore, profile, and clean the Titanic dataset, then split it and check your column
groups - no model yet. See `Lab5_Day1_Instructions.md` for the full walkthrough.

This notebook is just a shell: it gives you a place to write and document your work,
but the profiling, the decisions, and the reasoning are yours.

In [1]:
import pandas as pd
import numpy as np

from data import load_titanic

df, source = load_titanic()
print('source:', source)
df.head()


Loaded real Titanic from OpenML  (1309, 14)
source: openml


,Pclass,Survived,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,boat,body,home.dest
0,1,1,"Allen, Miss. Elisabeth Walton",female,29.0000,0,0,24160,211.3375,B5,S,2,NaN,"St Louis, MO"
1,1,1,"Allison, Master. Hudson Trevor",male,0.9167,1,2,113781,151.5500,C22 C26,S,11,NaN,"Montreal, PQ / Chesterville, ON"
2,1,0,"Allison, Miss. Helen Loraine",female,2.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
3,1,0,"Allison, Mr. Hudson Joshua Creighton",male,30.0000,1,2,113781,151.5500,C22 C26,S,NaN,135.0,"Montreal, PQ / Chesterville, ON"
4,1,0,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"


## Step 1: Load and profile

In [ ]:
# TODO: df.info(), df.describe(include='all').T, and missingness percentage per
# column, sorted descending
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 14 columns):
 #   Column     Non-Null Count  Dtype   
---  ------     --------------  -----   
 0   Pclass     1309 non-null   int64   
 1   Survived   1309 non-null   int32   
 2   Name       1309 non-null   object  
 3   Sex        1309 non-null   category
 4   Age        1046 non-null   float64 
 5   SibSp      1309 non-null   int64   
 6   Parch      1309 non-null   int64   
 7   Ticket     1309 non-null   object  
 8   Fare       1308 non-null   float64 
 9   Cabin      295 non-null    object  
 10  Embarked   1307 non-null   category
 11  boat       486 non-null    object  
 12  body       121 non-null    float64 
 13  home.dest  745 non-null    object  
dtypes: category(2), float64(3), int32(1), int64(3), object(5)
memory usage: 120.5+ KB


body         0.907563
Cabin        0.774637
boat         0.628724
home.dest    0.430863
Age          0.200917
Embarked     0.001528
Fare         0.000764
Pclass       0.000000
Survived     0.000000
Name         0.000000
Sex          0.000000
SibSp        0.000000
Parch        0.000000
Ticket       0.000000
dtype: float64

In [7]:
df.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Pclass,1309.0,NaN,NaN,NaN,2.294882,0.837836,1.0,2.0,3.0,3.0,3.0
Survived,1309.0,NaN,NaN,NaN,0.381971,0.486055,0.0,0.0,0.0,1.0,1.0
Name,1309,1307,"Connolly, Miss. Kate",2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Sex,1309,2,male,843,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Age,1046.0,NaN,NaN,NaN,29.881135,14.4135,0.1667,21.0,28.0,39.0,80.0
SibSp,1309.0,NaN,NaN,NaN,0.498854,1.041658,0.0,0.0,0.0,1.0,8.0
Parch,1309.0,NaN,NaN,NaN,0.385027,0.86556,0.0,0.0,0.0,0.0,9.0
Ticket,1309,929,CA. 2343,11,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Fare,1308.0,NaN,NaN,NaN,33.295479,51.758668,0.0,7.8958,14.4542,31.275,512.3292
Cabin,295,186,C23 C25 C27,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:

df.isnull().mean().sort_values(ascending=False)

body         0.907563
Cabin        0.774637
boat         0.628724
home.dest    0.430863
Age          0.200917
Embarked     0.001528
Fare         0.000764
Pclass       0.000000
Survived     0.000000
Name         0.000000
Sex          0.000000
SibSp        0.000000
Parch        0.000000
Ticket       0.000000
dtype: float64

### Step 1 observations

The dataset contains 1,309 rows and 14 columns. The missingness is not uniform across the dataset. `body`, `Cabin`, `boat`, `home.dest`, and `Age` have substantial missing values, while `Embarked` and `Fare` have only a very small amount of missingness.

This suggests that the missing values should not all be handled in the same way. For example, a column with around 90% missing values may not be useful in its raw form, whereas a column with only one or two missing values can usually be handled with a simple imputation. 

## Step 2: The most skipped checks - does missingness predict the target?

In [13]:
# TODO: for at least Age and Cabin, compare Survived rates between rows where the
# column is missing vs not. Is there a difference worth caring about?
for col in ['Age', 'Cabin']:
    print(f"\n {col} missingness vs survival:")
    results=(
        df.assign(missing=df[col].isnull())
        .groupby('missing')['Survived']
        .agg(['mean', 'count'])
        .rename(columns={'mean': 'Survival Rate', 'count': 'Count'})
    )
    results.index=['Available', 'Missing']
    display(results)


 Age missingness vs survival:


,Survival Rate,Count
Available,0.408222,1046
Missing,0.277567,263



 Cabin missingness vs survival:


,Survival Rate,Count
Available,0.654237,295
Missing,0.302761,1014


### Step 2 Observations
The survival rate differs between passengers with missing and non-missing values for both Age and Cabin.

For Age, passengers with an available age had a survival rate of about 40.82%, compared with 27.76% for passengers whose age was missing. This is a noticeable difference, so the missingness of Age may contain some information about the passengers. However, because Age is an important numerical feature and around 20% of its values are missing, I will keep the column and handle the missing values through imputation rather than dropping these rows.

The difference is much stronger for Cabin. Passengers with a recorded cabin had a survival rate of about 65.42%, while passengers with a missing cabin had a survival rate of only 30.28%. This suggests that cabin missingness carries useful information rather than being completely random.

Therefore, I will not simply discard all cabin information because of its high missingness. Instead, I will preserve the information about whether a cabin was recorded using a Cabin_known indicator, while dropping the raw Cabin column later because it is highly incomplete and contains detailed cabin identifiers.

## Step 3: Look for impossible values and placeholders

In [14]:
# TODO: check unique values in object columns, zero/negative fares, absurd ages -
# anything that looks like a placeholder rather than real missingness
for col in df.select_dtypes(include='object').columns:
    print(f"\nUnique values in {col}:")
    display(df[col].value_counts(dropna=False))
print("\nZero or negative fares:")
display(df[df['Fare'] <= 0][['Fare', 'Survived', 'Pclass', 'Embarked']])
print("\nAbsurd ages (negative or > 100):")
display(df[(df['Age'] < 0) | (df['Age'] > 100)][['Age', 'Survived', 'Pclass', 'Embarked']])
placeholder=['Unknown', 'T', '0', 'NA', 'N/A', 'na', 'n/a']
for col in df.select_dtypes(include='object').columns:
    if df[col].isin(placeholder).any():
        print(f"\nPlaceholder values in {col}:")
        display(df[df[col].isin(placeholder)][[col, 'Survived', 'Pclass', 'Embarked']])



Unique values in Name:


Name
Connolly, Miss. Kate             2
Kelly, Mr. James                 2
Allen, Miss. Elisabeth Walton    1
Ilmakangas, Miss. Ida Livija     1
Ilieff, Mr. Ylio                 1
                                ..
Hart, Miss. Eva Miriam           1
Harris, Mr. Walter               1
Harris, Mr. George               1
Harper, Rev. John                1
Zimmerman, Mr. Leo               1
Name: count, Length: 1307, dtype: int64


Unique values in Ticket:


Ticket
CA. 2343    11
1601         8
CA 2144      8
PC 17608     7
347077       7
            ..
373450       1
2223         1
350046       1
3101281      1
315082       1
Name: count, Length: 929, dtype: int64


Unique values in Cabin:


...

,Fare,Survived,Pclass,Embarked
7,0.0,0,1,S
70,0.0,0,1,S
125,0.0,0,1,S
150,0.0,0,1,S
170,0.0,1,1,S
223,0.0,0,1,S
234,0.0,0,1,S
363,0.0,0,2,S
384,0.0,0,2,S
410,0.0,0,2,S



Absurd ages (negative or > 100):


,Age,Survived,Pclass,Embarked



Placeholder values in Cabin:


,Cabin,Survived,Pclass,Embarked
30,T,0,1,S


### Step 3 Observations
The Age column contains no negative values or values above 100, so there are no obvious impossible ages that need to be removed.

The Fare column contains several observations with a fare of zero, but there are no negative fares. I will not automatically treat zero fares as invalid because a zero value can represent a genuine observation in the dataset. There is not enough evidence from this check alone to replace these values.

The object columns contain many unique values, particularly Name, Ticket, and home.dest, indicating that these are high-cardinality text/identifier-like features. I will therefore avoid using their raw values directly in the baseline pipeline.

The check for placeholder values identified T in the Cabin column. However, I will not automatically replace it with a missing value because a single unusual category is not sufficient evidence that it is a placeholder. Instead, the raw Cabin column will ultimately be removed because of its very high missingness, while the information about whether a cabin was recorded will be retained through the Cabin_known indicator.

## Step 4: Decide, and write it down

For each column with a problem, add a markdown cell (or a row in a table here) saying
what you did and why. Code with no commentary earns much less credit than the same
code with one sentence of justification.

**Cleaning decisions:**

## Cleaning decisions

| Column      | Problem                                  | Decision                                      | Why                                                                                                                                                                                                                                                                                                               |
| ----------- | ---------------------------------------- | --------------------------------------------- | ----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| `Age`       | About 20% missing                        | **Keep and impute later**                     | `Age` is a useful numerical feature, and dropping all rows with missing age would remove a significant portion of the data. I will impute the missing values later using a value calculated from the training data only.                                                                                          |
| `Cabin`     | About 77% missing                        | **Create `Cabin_known` and drop raw `Cabin`** | The missingness is informative: passengers with a recorded cabin had a much higher survival rate than passengers without one. Therefore, I want to preserve whether a cabin was recorded. However, the raw cabin values are highly incomplete and detailed, so I will use a simple missingness indicator instead. |
| `Embarked`  | Only 2 missing values                    | **Keep and impute later**                     | Very few values are missing, so there is no reason to discard those rows. Since this is a categorical feature, a simple most-frequent-category imputation is reasonable.                                                                                                                                          |
| `Fare`      | 1 missing value                          | **Keep and impute later**                     | Only one value is missing. Dropping the row is unnecessary, so I will keep the feature and use the training-set median for imputation. The median is a reasonable choice because `Fare` is skewed.                                                                                                                |
| `boat`      | About 63% missing                        | **Drop**                                      | This variable contains lifeboat information that is closely related to whether a passenger survived. It would not be a legitimate feature available before or at the time of prediction, so keeping it would introduce target leakage.                                                                            |
| `body`      | About 91% missing                        | **Drop**                                      | The `body` variable contains information associated with identifying the bodies of deceased passengers. This directly reveals outcome-related information and is therefore a serious target-leakage risk. It is also extremely incomplete.                                                                        |
| `home.dest` | About 43% missing and many unique values | **Drop**                                      | This is a high-cardinality text feature with substantial missingness. Using it would require additional text/location processing that is outside the scope of this baseline pipeline.                                                                                                                             |
| `Name`      | Very high cardinality                    | **Drop from direct features**                 | Passenger names are essentially free-text/identifier information. Using the raw names directly would create a large number of categories and is not appropriate for the simple baseline preprocessing pipeline.                                                                                                   |
| `Ticket`    | High-cardinality identifiers             | **Drop**                                      | Ticket numbers are identifiers rather than ordinary numerical measurements. Treating every ticket as a separate category would create many categories and is unnecessary for this baseline.                                                                                                                       |
| `Survived`  | Target variable                          | **Keep separately as `y`**                    | `Survived` is the outcome we want to predict, so it must not be included in the input feature matrix `X`.                                                                                                                                                                                                         |


### Target leakage check

I also checked whether any variables contain information that would only be available after the survival outcome was known.

The main leakage risks are `boat` and `body`. The `boat` variable contains lifeboat information, while `body` contains information related to deceased passengers. Both are strongly connected to the survival outcome and would not be appropriate predictors in a realistic prediction setting.

Therefore, I will exclude `boat` and `body` from the feature set. `Survived` will also be kept strictly as the target variable and will not be included in `X`.

For `Cabin`, I will preserve only whether the value was recorded through a `Cabin_known` indicator. This keeps potentially useful missingness information without relying on the highly incomplete raw cabin identifiers.


## Step 5: Split first, before fitting anything (`pipeline.py`)

In [19]:
# TODO: build X (drop the target and any obvious identifiers/free text you won't
# use directly) and y, then train_test_split with stratify=y and a fixed random_state.
# Nothing should be .fit() on the full dataset before this split exists.
from pipeline import train_test_split
df_clean=df.copy()
df['Cabin_known']=df['Cabin'].notna().astype(int)
X=df_clean.drop(columns=["Survived",
    "Cabin",
    "boat",
    "body",
    "home.dest",
    "Name",
    "Ticket"])
y=df_clean['Survived']
X_train, X_test, y_train, y_test=train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

print("\nTraining survival distribution:")
print(y_train.value_counts(normalize=True))

print("\nTest survival distribution:")
print(y_test.value_counts(normalize=True))

X_train shape: (1047, 8)
X_test shape: (262, 8)
y_train shape: (1047,)
y_test shape: (262,)

Training survival distribution:
Survived
0    0.617956
1    0.382044
Name: proportion, dtype: float64

Test survival distribution:
Survived
0    0.618321
1    0.381679
Name: proportion, dtype: float64



### Step 5 observations 

The dataset was split into **80% training data and 20% test data**, giving 1,047 training rows and 262 test rows.

There are **8 input features** in `X`. Seven of them come from the original dataset, while `Cabin_known` is an additional feature created from the `Cabin` column. Since the missingness of `Cabin` was found to be informative in Step 2, I kept this information as a simple indicator instead of using the raw `Cabin` column.

I used `stratify=y` so that the proportion of survived and non-survived passengers stays approximately the same in both sets. The training set has about **38.20% survivors**, while the test set has about **38.17% survivors**, showing that the split preserved the target distribution well.

I also used a fixed `random_state=0` so that the same train/test split can be reproduced later.

Most importantly, the split was performed **before fitting any imputer, encoder, scaler, or model**. This prevents information from the test set from influencing the preprocessing that will be learned from the training data.


## Step 6: Column groups - and check them by eye

In [22]:
# TODO: split X's columns into numeric vs categorical. Print both lists and look at
# them - does anything dtype-based selection picked up actually belong in the other
# group? (Think about what Pclass really represents.)
numeric_cols=X.select_dtypes(include=np.number).columns.tolist()
categorical_cols=X.select_dtypes(exclude=np.number).columns.tolist()
print("Initial Numeric columns:", numeric_cols)
print("Initial Categorical columns:", categorical_cols)
if "Pclass" in numeric_cols:
    numeric_cols.remove("Pclass")
    categorical_cols.append("Pclass")

print("\nFinal numeric columns:")
print(numeric_cols)

print("\nFinal categorical columns:")
print(categorical_cols)

Initial Numeric columns: ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'Cabin_known']
Initial Categorical columns: ['Sex', 'Embarked']

Final numeric columns:
['Age', 'SibSp', 'Parch', 'Fare', 'Cabin_known']

Final categorical columns:
['Sex', 'Embarked', 'Pclass']


### Step 6 observations 

I first used the column dtypes to create the numeric and categorical groups, but I then inspected the lists manually rather than relying on dtype alone.

`Pclass` is stored as an integer, so an automatic dtype-based split initially places it in the numeric group. However, `Pclass` represents passenger class (1st, 2nd, or 3rd class), not a continuous numerical measurement. Therefore, I moved `Pclass` to the categorical group.

The final numeric features are `Age`, `SibSp`, `Parch`, `Fare`, and `Cabin_known`. The categorical features are `Sex`, `Embarked`, and `Pclass`.

This manual check is important because the storage dtype does not always represent the semantic meaning of a feature.


---
**Before you close this notebook today:**
- Save your split so Day 2 resumes rather than re-derives it - e.g.
  `joblib.dump({"X_train": X_train, "X_test": X_test, "y_train": y_train, "y_test": y_test}, "split.joblib")`.
  Re-splitting tomorrow with a different `random_state` would silently invalidate every
  comparison you make against today's work.
- Confirm your split used `stratify=y` and a fixed `random_state`.
- Make sure Step 4's cleaning decisions are actually written down while the reasoning
  is fresh - reconstructing it tomorrow produces visibly thinner justifications.
- Keep this notebook and folder as-is. Day 2 is a new notebook here, not a restart.

In [23]:
import joblib

joblib.dump(
    {
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test
    },
    "split.joblib"
)

print("Split saved successfully to split.joblib")

Split saved successfully to split.joblib
